In [ ]:
# Install Unsloth: It makes fine-tuning 2x faster and uses 70% less memory.
!pip install unsloth
# Install supporting libraries for handling the model and data.
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load the base Llama-3 model in 4-bit to save GPU memory.
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/llama-3-8b-bnb-4bit",
#     max_seq_length = 2048,
#     load_in_4bit = True,
# )
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,          # auto
    load_in_4bit = True    # saves memory
)


In [ ]:

# Add 'LoRA' adapters: Instead of training the whole model,
# Standard LoRA setup as before
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

In [ ]:
import json
from datasets import Dataset

# Load your uploaded file.
with open('companies_dataset.json', 'r') as f:
    raw_data = json.load(f)

# This template tells the model how to read your data.
prompt_template = """Instruction: Provide information about the company.
Company Name: {}
Response: {}"""

def format_data(example):
    # We combine the company details into a single response block.
    response_text = f"URL: {example['URL']}, Category: {example['Category']}, Description: {example['Description']}"
    return { "text" : prompt_template.format(example['Name'], response_text) }

# Convert JSON to a format ready for training.
dataset = Dataset.from_list(raw_data).map(format_data)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2, # Number of rows processed at once.
        gradient_accumulation_steps = 4, # Simulates a larger batch for stability.
        max_steps = 500,                # How many "lessons" to run.
        learning_rate = 2e-4,           # The speed of learning.
        fp16 = True,                    # Uses faster math (Half-precision).
        logging_steps = 1,               # Show loss for every single step.
        output_dir = "outputs",
    ),
)

trainer.train() # This starts the actual fine-tuning.

In [ ]:
# Save locally first
model.save_pretrained("saas_model_lora")
tokenizer.save_pretrained("saas_model_lora")

# Export to GGUF (Quantized to q8_0 for high quality)
model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method = "q8_0")

In [ ]:
# Cell 10: Zip and download the LoRA adapter
!zip -r saas_model_lora.zip saas_model_lora/

# Download to your computer
from google.colab import files
files.download("saas_model_lora.zip")
files.download("model_gguf_gguf/Qwen3-0.6B.Q8_0.gguf")
files.download("model_gguf_gguf/Modelfile")
print("✅ Download started! Check your browser downloads.")